In [1]:
# ============================================================
# COMPLETE JUPYTER NOTEBOOK
# K-MEANS + NLP ANALYSIS
# Dataset: metric-node-192.168.31.103:9100.csv
# ============================================================

# ============================================================
# CELL 1 — Install required packages
# ============================================================

!pip install -q pandas numpy scikit-learn matplotlib seaborn nltk scipy


# ============================================================
# CELL 2 — Import libraries
# ============================================================

import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack, csr_matrix

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import nltk
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

# Download English stopwords
nltk.download("stopwords", quiet=True)

print("Libraries loaded successfully.")


# ============================================================
# CELL 3 — Configuration
# ============================================================

FILE_PATH = "metric-node-192.168.31.103:9100.csv"

OUTPUT_FILE = "metric-node-192.168.31.103-9100-clustered.csv"

RANDOM_STATE = 42

# Maximum number of TF-IDF features
MAX_TFIDF_FEATURES = 5000

# K values to test
MIN_K = 2
MAX_K = 10

# Maximum points used for PCA visualization
MAX_PCA_POINTS = 10000


# ============================================================
# CELL 4 — Check that the dataset exists
# ============================================================

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{FILE_PATH}\n\n"
        "Put the CSV file in the same folder as this Jupyter notebook."
    )

print(f"Dataset found: {FILE_PATH}")


# ============================================================
# CELL 5 — Load CSV
# ============================================================

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print()
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


# ============================================================
# CELL 6 — Dataset information
# ============================================================

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

df.info()


# ============================================================
# CELL 7 — Column names
# ============================================================

print("=" * 70)
print("COLUMNS")
print("=" * 70)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:3}. {column}")


# ============================================================
# CELL 8 — Missing values
# ============================================================

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isnull().sum().values,
    "missing_percent": (
        df.isnull().sum().values / len(df) * 100
    )
})

missing = missing.sort_values(
    "missing_count",
    ascending=False
)

display(missing)


# ============================================================
# CELL 9 — Duplicate records
# ============================================================

duplicates = df.duplicated().sum()

print("Duplicate rows:", duplicates)

if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")


# ============================================================
# CELL 10 — Identify numeric and text columns
# ============================================================

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

text_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("=" * 70)
print("NUMERIC COLUMNS")
print("=" * 70)

print(numeric_columns)

print()
print("=" * 70)
print("TEXT COLUMNS")
print("=" * 70)

print(text_columns)


# ============================================================
# CELL 11 — Convert numeric-looking object columns
# ============================================================

data = df.copy()

for column in data.columns:

    if data[column].dtype == "object":

        converted = pd.to_numeric(
            data[column],
            errors="coerce"
        )

        # If most values can be converted to numbers,
        # treat this column as numeric.
        valid_ratio = converted.notna().mean()

        if valid_ratio >= 0.80:
            data[column] = converted

# Recalculate column types
numeric_columns = data.select_dtypes(
    include=[np.number]
).columns.tolist()

text_columns = data.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

print()
print("Text columns:")
print(text_columns)


# ============================================================
# CELL 12 — Clean numeric data
# ============================================================

for column in numeric_columns:

    # Replace infinity values
    data[column] = data[column].replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Replace missing values with median
    median_value = data[column].median()

    if pd.isna(median_value):
        median_value = 0

    data[column] = data[column].fillna(
        median_value
    )


# ============================================================
# CELL 13 — Clean text data
# ============================================================

for column in text_columns:

    data[column] = (
        data[column]
        .fillna("")
        .astype(str)
    )


# ============================================================
# CELL 14 — Remove constant numeric columns
# ============================================================

numeric_features = data[numeric_columns].copy()

constant_columns = [
    column
    for column in numeric_features.columns
    if numeric_features[column].nunique() <= 1
]

if constant_columns:

    print("Removing constant columns:")
    print(constant_columns)

    numeric_features = numeric_features.drop(
        columns=constant_columns
    )

else:

    print("No constant numeric columns found.")


# ============================================================
# CELL 15 — Numerical dataset statistics
# ============================================================

print("=" * 70)
print("NUMERICAL STATISTICS")
print("=" * 70)

display(
    numeric_features.describe().T
)


# ============================================================
# CELL 16 — Correlation matrix
# ============================================================

if numeric_features.shape[1] >= 2:

    plt.figure(
        figsize=(14, 10)
    )

    correlation = numeric_features.corr()

    sns.heatmap(
        correlation,
        cmap="coolwarm",
        center=0,
        linewidths=0.2
    )

    plt.title(
        "Correlation Matrix"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# CELL 17 — Numerical feature distributions
# ============================================================

if numeric_features.shape[1] > 0:

    numeric_features.hist(
        figsize=(16, 12),
        bins=30
    )

    plt.suptitle(
        "Numerical Feature Distributions",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# CELL 18 — Standardize numerical features
# ============================================================

if numeric_features.shape[1] > 0:

    scaler = StandardScaler()

    X_numeric = scaler.fit_transform(
        numeric_features
    )

    print(
        "Standardized numerical matrix:",
        X_numeric.shape
    )

else:

    X_numeric = np.empty(
        (len(data), 0)
    )

    print("No numerical features available.")


# ============================================================
# CELL 19 — Create combined text for NLP
# ============================================================

if len(text_columns) > 0:

    data["combined_text"] = (
        data[text_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

else:

    data["combined_text"] = ""


# ============================================================
# CELL 20 — NLP text cleaning
# ============================================================

def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Replace underscores and separators
    text = re.sub(
        r"[_:/\\\-]+",
        " ",
        text
    )

    # Keep letters, numbers and spaces
    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


data["clean_text"] = (
    data["combined_text"]
    .apply(clean_text)
)

display(
    data[
        ["combined_text", "clean_text"]
    ].head(10)
)


# ============================================================
# CELL 21 — TF-IDF NLP
# ============================================================

stop_words = stopwords.words("english")

usable_text = data["clean_text"].str.strip()

if usable_text.ne("").any():

    vectorizer = TfidfVectorizer(
        stop_words=stop_words,
        max_features=MAX_TFIDF_FEATURES,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True
    )

    X_text = vectorizer.fit_transform(
        usable_text
    )

    print(
        "TF-IDF matrix:",
        X_text.shape
    )

    print(
        "Number of NLP features:",
        len(vectorizer.get_feature_names_out())
    )

else:

    X_text = None

    print(
        "No usable text was found. "
        "NLP features will not be used."
    )


# ============================================================
# CELL 22 — Combine numeric + NLP features
# ============================================================

if X_text is not None:

    X_numeric_sparse = csr_matrix(
        X_numeric
    )

    X = hstack(
        [
            X_numeric_sparse,
            X_text
        ]
    ).tocsr()

else:

    X = csr_matrix(
        X_numeric
    )

print("=" * 70)
print("FINAL FEATURE MATRIX")
print("=" * 70)

print("Rows:", X.shape[0])
print("Features:", X.shape[1])


# ============================================================
# CELL 23 — Elbow Method
# ============================================================

max_possible_k = min(
    MAX_K,
    len(data) - 1
)

if max_possible_k < MIN_K:

    raise ValueError(
        "Dataset is too small for K-Means."
    )

k_values = list(
    range(
        MIN_K,
        max_possible_k + 1
    )
)

inertias = []

print("=" * 70)
print("ELBOW METHOD")
print("=" * 70)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    model.fit(X)

    inertias.append(
        model.inertia_
    )

    print(
        f"K = {k:2d} | "
        f"Inertia = {model.inertia_:.4f}"
    )


# ============================================================
# CELL 24 — Plot Elbow Curve
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    k_values,
    inertias,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "K-Means Elbow Method"
)

plt.xticks(k_values)
plt.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# CELL 25 — Silhouette Score
# ============================================================

silhouette_scores = []

print("=" * 70)
print("SILHOUETTE ANALYSIS")
print("=" * 70)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )

    labels = model.fit_predict(X)

    # Silhouette score can be expensive for very
    # large datasets, so sample when necessary.
    if len(data) > 10000:

        rng = np.random.RandomState(
            RANDOM_STATE
        )

        sample_indices = rng.choice(
            len(data),
            10000,
            replace=False
        )

        score = silhouette_score(
            X[sample_indices],
            labels[sample_indices]
        )

    else:

        score = silhouette_score(
            X,
            labels
        )

    silhouette_scores.append(
        score
    )

    print(
        f"K = {k:2d} | "
        f"Silhouette Score = {score:.4f}"
    )


# ============================================================
# CELL 26 — Plot Silhouette Scores
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "K-Means Silhouette Analysis"
)

plt.xticks(k_values)
plt.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# CELL 27 — Select best K
# ============================================================

best_index = np.argmax(
    silhouette_scores
)

best_k = k_values[
    best_index
]

best_score = silhouette_scores[
    best_index
]

print("=" * 70)
print("BEST K")
print("=" * 70)

print(
    f"Best K: {best_k}"
)

print(
    f"Silhouette Score: {best_score:.4f}"
)


# ============================================================
# CELL 28 — Train final K-Means
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=20
)

clusters = kmeans.fit_predict(
    X
)

data["cluster"] = clusters

print(
    "K-Means clustering completed."
)


# ============================================================
# CELL 29 — Cluster counts
# ============================================================

cluster_counts = (
    data["cluster"]
    .value_counts()
    .sort_index()
)

print("=" * 70)
print("CLUSTER DISTRIBUTION")
print("=" * 70)

display(
    cluster_counts.to_frame(
        name="record_count"
    )
)


# ============================================================
# CELL 30 — Cluster distribution plot
# ============================================================

plt.figure(
    figsize=(9, 6)
)

sns.barplot(
    x=cluster_counts.index.astype(str),
    y=cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "K-Means Cluster Distribution"
)

plt.tight_layout()
plt.show()


# ============================================================
# CELL 31 — PCA visualization
# ============================================================

if len(data) > MAX_PCA_POINTS:

    rng = np.random.RandomState(
        RANDOM_STATE
    )

    sample_indices = rng.choice(
        len(data),
        MAX_PCA_POINTS,
        replace=False
    )

    X_pca_input = X[
        sample_indices
    ]

    labels_pca = clusters[
        sample_indices
    ]

else:

    X_pca_input = X
    labels_pca = clusters


# Convert sparse -> dense for PCA
X_pca_dense = X_pca_input.toarray()

pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)

X_pca = pca.fit_transform(
    X_pca_dense
)

print("=" * 70)
print("PCA")
print("=" * 70)

print(
    "PC1 explained variance:",
    round(
        pca.explained_variance_ratio_[0],
        4
    )
)

print(
    "PC2 explained variance:",
    round(
        pca.explained_variance_ratio_[1],
        4
    )
)


# ============================================================
# CELL 32 — Plot PCA clusters
# ============================================================

plt.figure(
    figsize=(12, 8)
)

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=labels_pca,
    palette="tab10",
    s=50,
    alpha=0.75
)

plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.title(
    "K-Means Clusters — PCA Visualization"
)

plt.legend(
    title="Cluster"
)

plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


# ============================================================
# CELL 33 — Numerical metrics by cluster
# ============================================================

if len(numeric_features.columns) > 0:

    cluster_numeric_mean = (
        data.groupby("cluster")[
            numeric_features.columns
        ]
        .mean()
    )

    print("=" * 70)
    print("AVERAGE NUMERICAL METRICS BY CLUSTER")
    print("=" * 70)

    display(
        cluster_numeric_mean
    )


# ============================================================
# CELL 34 — Median numerical metrics by cluster
# ============================================================

if len(numeric_features.columns) > 0:

    cluster_numeric_median = (
        data.groupby("cluster")[
            numeric_features.columns
        ]
        .median()
    )

    print("=" * 70)
    print("MEDIAN NUMERICAL METRICS BY CLUSTER")
    print("=" * 70)

    display(
        cluster_numeric_median
    )


# ============================================================
# CELL 35 — Standardized cluster profile
# ============================================================

if len(numeric_features.columns) > 0:

    profile = pd.DataFrame(
        X_numeric,
        columns=numeric_features.columns
    )

    profile["cluster"] = clusters

    cluster_profile = (
        profile
        .groupby("cluster")
        .mean()
    )

    print("=" * 70)
    print("STANDARDIZED CLUSTER PROFILE")
    print("=" * 70)

    display(
        cluster_profile
    )


# ============================================================
# CELL 36 — Heatmap of cluster profiles
# ============================================================

if len(numeric_features.columns) > 0:

    plt.figure(
        figsize=(16, 8)
    )

    sns.heatmap(
        cluster_profile.T,
        cmap="coolwarm",
        center=0,
        annot=False
    )

    plt.xlabel(
        "Cluster"
    )

    plt.ylabel(
        "Metric"
    )

    plt.title(
        "Standardized Metric Profile by Cluster"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# CELL 37 — Most important NLP terms per cluster
# ============================================================

if X_text is not None:

    feature_names = (
        vectorizer
        .get_feature_names_out()
    )

    print("=" * 70)
    print("TOP NLP TERMS PER CLUSTER")
    print("=" * 70)

    top_terms_by_cluster = {}

    for cluster_id in sorted(
        data["cluster"].unique()
    ):

        indices = np.where(
            clusters == cluster_id
        )[0]

        cluster_text = X_text[
            indices
        ]

        mean_tfidf = np.asarray(
            cluster_text.mean(
                axis=0
            )
        ).flatten()

        top_indices = (
            mean_tfidf
            .argsort()[-20:][::-1]
        )

        terms = []

        for index in top_indices:

            if mean_tfidf[index] > 0:

                terms.append(
                    (
                        feature_names[index],
                        mean_tfidf[index]
                    )
                )

        top_terms_by_cluster[
            cluster_id
        ] = terms

        print(
            f"\nCLUSTER {cluster_id}"
        )

        for term, score in terms:

            print(
                f"{term:<40} "
                f"{score:.4f}"
            )


# ============================================================
# CELL 38 — NLP terms dataframe
# ============================================================

if X_text is not None:

    nlp_rows = []

    for cluster_id, terms in (
        top_terms_by_cluster.items()
    ):

        for rank, (
            term,
            score
        ) in enumerate(
            terms,
            start=1
        ):

            nlp_rows.append({
                "cluster": cluster_id,
                "rank": rank,
                "term": term,
                "tfidf_score": score
            })

    nlp_summary = pd.DataFrame(
        nlp_rows
    )

    display(
        nlp_summary
    )


# ============================================================
# CELL 39 — Example records from each cluster
# ============================================================

print("=" * 70)
print("EXAMPLE RECORDS FROM EACH CLUSTER")
print("=" * 70)

display_columns = [
    column
    for column in data.columns
    if column not in [
        "combined_text",
        "clean_text"
    ]
]

for cluster_id in sorted(
    data["cluster"].unique()
):

    print()
    print("=" * 70)
    print(
        f"CLUSTER {cluster_id}"
    )
    print("=" * 70)

    cluster_records = (
        data[
            data["cluster"] == cluster_id
        ][display_columns]
        .head(10)
    )

    display(
        cluster_records
    )


# ============================================================
# CELL 40 — Detect high/low metric cluster characteristics
# ============================================================

if len(numeric_features.columns) > 0:

    print("=" * 70)
    print("CLUSTER CHARACTERISTICS")
    print("=" * 70)

    global_mean = numeric_features.mean()
    global_std = numeric_features.std()

    for cluster_id in sorted(
        data["cluster"].unique()
    ):

        cluster_mean = (
            data[
                data["cluster"] == cluster_id
            ][numeric_features.columns]
            .mean()
        )

        z_scores = (
            (cluster_mean - global_mean)
            / global_std.replace(
                0,
                np.nan
            )
        )

        high_metrics = (
            z_scores[
                z_scores > 1
            ]
            .sort_values(
                ascending=False
            )
        )

        low_metrics = (
            z_scores[
                z_scores < -1
            ]
            .sort_values()
        )

        print()
        print(
            f"CLUSTER {cluster_id}"
        )

        if len(high_metrics) > 0:

            print(
                "Higher-than-average metrics:"
            )

            for metric, value in (
                high_metrics.head(10).items()
            ):

                print(
                    f"  {metric}: "
                    f"z={value:.2f}"
                )

        else:

            print(
                "No strongly elevated metrics."
            )

        if len(low_metrics) > 0:

            print(
                "Lower-than-average metrics:"
            )

            for metric, value in (
                low_metrics.head(10).items()
            ):

                print(
                    f"  {metric}: "
                    f"z={value:.2f}"
                )

        else:

            print(
                "No strongly reduced metrics."
            )


# ============================================================
# CELL 41 — Calculate cluster sizes as percentages
# ============================================================

cluster_distribution = (
    data["cluster"]
    .value_counts()
    .sort_index()
    .to_frame(
        name="records"
    )
)

cluster_distribution[
    "percentage"
] = (
    cluster_distribution["records"]
    / len(data)
    * 100
)

print("=" * 70)
print("CLUSTER DISTRIBUTION (%)")
print("=" * 70)

display(
    cluster_distribution
)


# ============================================================
# CELL 42 — Optional anomaly-style analysis
# ============================================================
#
# IMPORTANT:
# K-Means is clustering, not a formal anomaly detector.
#
# This section identifies records that are far from their
# assigned cluster centroid as potentially unusual.
# ============================================================

distances = kmeans.transform(X)

assigned_cluster_distance = (
    distances[
        np.arange(len(data)),
        clusters
    ]
)

data["cluster_distance"] = (
    assigned_cluster_distance
)

# 95th percentile threshold
distance_threshold = np.percentile(
    assigned_cluster_distance,
    95
)

data["potential_anomaly"] = (
    data["cluster_distance"]
    > distance_threshold
)

print("=" * 70)
print("POTENTIAL ANOMALY ANALYSIS")
print("=" * 70)

print(
    "95th percentile distance threshold:",
    distance_threshold
)

print(
    "Potential anomalies:",
    data["potential_anomaly"].sum()
)

print(
    "Potential anomaly percentage:",
    round(
        data["potential_anomaly"].mean() * 100,
        2
    ),
    "%"
)


# ============================================================
# CELL 43 — Display potential anomalies
# ============================================================

anomalies = (
    data[
        data["potential_anomaly"]
    ]
    .sort_values(
        "cluster_distance",
        ascending=False
    )
)

print("=" * 70)
print("TOP POTENTIAL ANOMALIES")
print("=" * 70)

display(
    anomalies.head(50)
)


# ============================================================
# CELL 44 — Cluster distance distribution
# ============================================================

plt.figure(
    figsize=(10, 6)
)

sns.histplot(
    data["cluster_distance"],
    bins=50,
    kde=True
)

plt.axvline(
    distance_threshold,
    linestyle="--",
    linewidth=2,
    label="95th percentile"
)

plt.xlabel(
    "Distance from Assigned Cluster Centroid"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "K-Means Cluster Distance Distribution"
)

plt.legend()

plt.tight_layout()
plt.show()


# ============================================================
# CELL 45 — Save final clustered dataset
# ============================================================

# Remove temporary NLP columns from saved dataset
columns_to_remove = [
    "combined_text",
    "clean_text"
]

final_data = data.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in data.columns
    ]
)

final_data.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("FILE SAVED")
print("=" * 70)

print(
    os.path.abspath(
        OUTPUT_FILE
    )
)


# ============================================================
# CELL 46 — Save NLP summary
# ============================================================

if X_text is not None:

    NLP_OUTPUT_FILE = (
        "metric-node-192.168.31.103-9100-nlp-summary.csv"
    )

    nlp_summary.to_csv(
        NLP_OUTPUT_FILE,
        index=False
    )

    print(
        "NLP summary saved to:",
        os.path.abspath(
            NLP_OUTPUT_FILE
        )
    )


# ============================================================
# CELL 47 — Save cluster statistics
# ============================================================

CLUSTER_OUTPUT_FILE = (
    "metric-node-192.168.31.103-9100-cluster-summary.csv"
)

cluster_distribution.to_csv(
    CLUSTER_OUTPUT_FILE
)

print(
    "Cluster summary saved to:",
    os.path.abspath(
        CLUSTER_OUTPUT_FILE
    )
)


# ============================================================
# CELL 48 — Final report
# ============================================================

print()
print("=" * 80)
print("FINAL K-MEANS + NLP REPORT")
print("=" * 80)

print(
    f"Input file: {FILE_PATH}"
)

print(
    f"Number of records: {len(data):,}"
)

print(
    f"Numerical features: {X_numeric.shape[1]}"
)

if X_text is not None:

    print(
        f"NLP/TF-IDF features: {X_text.shape[1]}"
    )

else:

    print(
        "NLP/TF-IDF features: 0"
    )

print(
    f"Selected K: {best_k}"
)

print(
    f"Best silhouette score: {best_score:.4f}"
)

print(
    f"Potential anomalies: "
    f"{data['potential_anomaly'].sum():,}"
)

print(
    f"Potential anomaly rate: "
    f"{data['potential_anomaly'].mean() * 100:.2f}%"
)

print()
print("Cluster sizes:")

for cluster_id, count in (
    cluster_counts.items()
):

    percentage = (
        count / len(data) * 100
    )

    print(
        f"  Cluster {cluster_id}: "
        f"{count:,} records "
        f"({percentage:.2f}%)"
    )

print()
print("=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

Libraries loaded successfully.


FileNotFoundError: Dataset not found:
metric-node-192.168.31.103:9100.csv

Put the CSV file in the same folder as this Jupyter notebook.